# Phase 6 — Explainability

A score of 0.88 tells a retention agent nothing they can act on. SHAP turns it into
"month-to-month contract, two months tenure, no tech support" — which is a phone call.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np
pd.set_option('display.width', 120)

from config import REPORTS_DIR, DB_PATH
from explain import load_pipeline, get_explainer, explain_customer
pipe = load_pipeline()
explainer, pre, kind = get_explainer(pipe)
print('explainer type:', kind)

explainer type: tree


## Global drivers

In [2]:
pd.read_csv(REPORTS_DIR/'shap_importance.csv')

,feature,mean_abs_shap
0,month-to-month contract,0.0509
1,contract length,0.0505
2,fiber optic internet,0.0314
3,months as a customer,0.0297
4,no online security,0.0272
5,two-year contract,0.0253
6,lifecycle stage,0.0198
7,no tech support add-on,0.0197
8,first year of service,0.0180
9,total billed to date,0.0155


Contract dominates, then internet product, then tenure. This agrees with the SQL and
EDA phases — a model whose explanations contradicted the exploratory analysis would
be a signal that something upstream is broken.

## Per-customer explanations

In [3]:
df = pd.read_sql_query('SELECT * FROM v_customer_360', sqlite3.connect(DB_PATH))
for cid in ['3668-QPYBK', '7590-VHVEG']:
    c = df[df.customer_id == cid].iloc[0].to_dict()
    r = explain_customer(c, pipe, explainer, pre, kind)
    print(f"\n{cid}  actual={c['churn']}  contract={c['contract']}  tenure={c['tenure']}")
    print(f"  predicted risk: {r['churn_probability']:.1%}")
    for f in r['risk_factors']:
        print(f"    + {f['label']:<42}{f['impact']:+.3f}")
    for f in r['protective_factors']:
        print(f"    - {f['label']:<42}{f['impact']:+.3f}")
    print(f"  action: {r['recommended_action'][:88]}...")


3668-QPYBK  actual=Yes  contract=Month-to-month  tenure=2
  predicted risk: 63.5%
    + months as a customer                      +0.044
    + month-to-month contract                   +0.044
    + new + month-to-month + no tech support    +0.039
    - has online security                       -0.014
    - DSL internet                              -0.013
    - protection add-ons held                   -0.006
  action: Offer a 12-month contract with the first two months discounted — contract length is the ...

7590-VHVEG  actual=No  contract=Month-to-month  tenure=1
  predicted risk: 74.0%
    + months as a customer                      +0.047
    + month-to-month contract                   +0.043
    + new + month-to-month + no tech support    +0.036
    - DSL internet                              -0.018
    - has online backup                         -0.007
    - has a partner on account                  -0.004
  action: Offer a 12-month contract with the first two months discounted 

Note the second customer: predicted 74% risk, actually stayed. That is a false
positive, and showing one is deliberate — at a 0.31 threshold the model is tuned to
over-flag, and an agent who never sees a false positive does not understand the tool
they are using. The cost of that error is one retention offer.

## How this reaches the browser

In [4]:
import inspect, explain
print(inspect.getsource(explain.explain_customer)[:1500])

def explain_customer(customer: dict, pipeline, explainer, pre, kind: str,
                     top_n: int = 3) -> dict:
    """Score one customer and return probability + ranked human reasons."""
    row = pd.DataFrame([customer])
    feats = engineer_features(row)
    proba = float(pipeline.predict_proba(feats)[:, 1][0])

    matrix = pre.transform(feats)
    names = list(pre.get_feature_names_out())
    values = shap_values_for(explainer, matrix, kind)[0]
    encoded = np.asarray(matrix)[0]

    order = np.argsort(np.abs(values))[::-1]
    risk_factors, protective_factors = [], []
    seen: set[str] = set()

    for idx in order:
        name = names[idx]
        label = _prettify(name)

        # A one-hot column sitting at 0 means the customer does NOT have
        # that attribute. Its SHAP value is real, but showing an agent
        # "fiber optic internet" for a DSL customer is worse than showing
        # nothing, so inactive dummies are dropped from the narrative.
        # Ac

`explain_customer` is imported directly by `backend/app/service.py`, so the
explanation an agent reads in the browser is the same computation audited here —
no second implementation to drift out of sync.